# Temperature Prediction Exploration

This notebook follows the same logic as the current app code in `src/`.

Main goals:

- avoid data leakage
- train on past data
- test on future data
- understand why predictions can go flat or down
- use the same `SimpleRNN(..., activation="relu")` model as the app

## 1. Import Libraries

These are the same main libraries used by the app.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, SimpleRNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

## 2. Set Project Paths

This works if you open the notebook from the `notebooks/` folder or from the project root.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_FILE = PROJECT_ROOT / "data" / "temperature.csv"
MODEL_FILE = PROJECT_ROOT / "models" / "temperature_rnn.keras"
SCALER_FILE = PROJECT_ROOT / "models" / "temperature_scaler.npz"

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_FILE)

## 3. Load the Data

The CSV has one simple time series: temperature by day.

In [ ]:
data_frame = pd.read_csv(DATA_FILE)
data_frame.head()

In [ ]:
temperatures = data_frame["temperature"].to_numpy(dtype=np.float32)

print("Total temperature values:", len(temperatures))
print(temperatures.astype(int))

## 4. Plot the Raw Data

Always look at the data before training. This data is perfectly increasing by 1 each day.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(data_frame["day"], data_frame["temperature"], marker="o")
plt.xlabel("Day")
plt.ylabel("Temperature")
plt.title("Temperature by Day")
plt.grid(True)
plt.show()

## 5. Helper Functions

These functions match the beginner-friendly logic in `src/data_utils.py`.

The most important function is `make_sequences()`. It turns this:

```text
20, 21, 22, 23, 24, 25
```

into this training example:

```text
[20, 21, 22, 23, 24] -> 25
```

In [ ]:
def split_temperatures(values, test_size=0.2):
    split_index = int(len(values) * (1 - test_size))
    train_values = values[:split_index]
    test_values = values[split_index:]
    return train_values, test_values


def make_sequences(values, sequence_length):
    X = []
    y = []

    for i in range(len(values) - sequence_length):
        input_sequence = values[i : i + sequence_length]
        next_value = values[i + sequence_length]

        X.append(input_sequence)
        y.append(next_value)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, y


def scale_values(values, mean, standard_deviation):
    return (np.asarray(values, dtype=np.float32) - mean) / standard_deviation


def unscale_values(values, mean, standard_deviation):
    return (np.asarray(values, dtype=np.float32) * standard_deviation) + mean

## 6. Split Data Without Leakage

For time-series data, do not shuffle before splitting.

We train on the earlier values and test on the later values. This is closer to real prediction, where future data is not known yet.

In [ ]:
SEQUENCE_LENGTH = 5
TEST_SIZE = 0.2

train_temperatures, test_temperatures = split_temperatures(
    temperatures,
    test_size=TEST_SIZE,
)

print("Total values:", len(temperatures))
print("Train values:", len(train_temperatures), train_temperatures.astype(int).tolist())
print("Test values:", len(test_temperatures), test_temperatures.astype(int).tolist())

## 7. Scale Using Training Data Only

This is where data leakage often happens.

Wrong approach:

```text
calculate mean and standard deviation from all data
```

Correct approach:

```text
calculate mean and standard deviation from training data only
```

Then use those same training numbers to scale validation, test, and prediction input.

In [ ]:
mean = np.float32(train_temperatures.mean())
standard_deviation = np.float32(train_temperatures.std())

scaled_train = scale_values(train_temperatures, mean, standard_deviation)

print("Training mean:", mean)
print("Training standard deviation:", standard_deviation)
print("First scaled train values:", scaled_train[:5])

## 8. Create Training Sequences

Now we make the examples that the RNN learns from.

In [ ]:
X_train, y_train = make_sequences(scaled_train, SEQUENCE_LENGTH)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nFirst 5 examples before scaling, easier to read:")
X_raw, y_raw = make_sequences(train_temperatures, SEQUENCE_LENGTH)
for i in range(5):
    print("Input:", X_raw[i].flatten().astype(int).tolist(), "Answer:", int(y_raw[i]))

## 9. Build the Current App Model

The app now uses `activation="relu"` in the RNN layer.

Why? The default `SimpleRNN` activation is `tanh`. On this tiny always-rising dataset, `tanh` can flatten predictions when values move above the range seen during training. `relu` works better for this beginner trend example.

In [ ]:
def create_model(sequence_length=5, rnn_units=32):
    model = Sequential()
    model.add(Input(shape=(sequence_length, 1)))
    model.add(SimpleRNN(rnn_units, activation="relu"))
    model.add(Dense(1))

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
    return model


model = create_model(sequence_length=SEQUENCE_LENGTH, rnn_units=32)
model.summary()

## 10. Train the Model

Important time-series setting:

```python
shuffle=False
```

This keeps the order of examples stable.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True,
    verbose=1,
)

save_best_model = ModelCheckpoint(
    MODEL_FILE,
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

history = model.fit(
    X_train,
    y_train,
    epochs=300,
    batch_size=8,
    validation_split=0.2,
    shuffle=False,
    callbacks=[early_stopping, save_best_model],
    verbose=1,
)

## 11. Save Scaling Numbers

The model was trained on scaled values, so later predictions must use the same scaling numbers.

In [ ]:
MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    SCALER_FILE,
    mean=mean,
    standard_deviation=standard_deviation,
    sequence_length=SEQUENCE_LENGTH,
)

print("Model saved to:", MODEL_FILE)
print("Scaling numbers saved to:", SCALER_FILE)

## 12. Plot Training Loss

Loss should generally go down. Validation loss checks how well the model handles examples it did not directly train on.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

best_epoch = history.history["val_loss"].index(min(history.history["val_loss"])) + 1
print("Best validation loss:", min(history.history["val_loss"]))
print("Best epoch:", best_epoch)

## 13. Evaluate on Test Data Only

This matches the default behavior in `src/evaluate.py`.

We use the last training temperatures only as context. The actual predictions are for test temperatures only.

In [ ]:
context = train_temperatures[-SEQUENCE_LENGTH:]
evaluation_temperatures = list(context) + list(test_temperatures)

scaled_evaluation_temperatures = scale_values(
    evaluation_temperatures,
    mean,
    standard_deviation,
)

X_test, y_test = make_sequences(scaled_evaluation_temperatures, SEQUENCE_LENGTH)

scaled_predictions = model.predict(X_test, verbose=0).flatten()

actual_temperatures = unscale_values(y_test, mean, standard_deviation)
predicted_temperatures = unscale_values(scaled_predictions, mean, standard_deviation)

errors = np.abs(actual_temperatures - predicted_temperatures)
mean_absolute_error = errors.mean()

print(f"Mean absolute error: {mean_absolute_error:.2f}")

comparison = pd.DataFrame({
    "actual": actual_temperatures,
    "predicted": predicted_temperatures,
    "error": errors,
})
comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(actual_temperatures, marker="o", label="Actual")
plt.plot(predicted_temperatures, marker="o", label="Predicted")
plt.xlabel("Test Sequence")
plt.ylabel("Temperature")
plt.title("RNN Temperature Prediction - Test Data Only")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Optional: Evaluate All Data

This is useful for inspection, but it is not a true future-data test because it includes training rows.

In [ ]:
scaled_all_temperatures = scale_values(temperatures, mean, standard_deviation)
X_all, y_all = make_sequences(scaled_all_temperatures, SEQUENCE_LENGTH)

all_scaled_predictions = model.predict(X_all, verbose=0).flatten()
all_actual = unscale_values(y_all, mean, standard_deviation)
all_predicted = unscale_values(all_scaled_predictions, mean, standard_deviation)

plt.figure(figsize=(10, 5))
plt.plot(all_actual, label="Actual")
plt.plot(all_predicted, label="Predicted")
plt.xlabel("Sequence")
plt.ylabel("Temperature")
plt.title("RNN Temperature Prediction - All Data")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Predict the Next Temperature

This matches `src/predict.py`.

The input must have the same length as `SEQUENCE_LENGTH`.

In [ ]:
new_temperatures = np.array([45, 46, 47, 48, 49], dtype=np.float32)

scaled_input = scale_values(new_temperatures, mean, standard_deviation)
model_input = scaled_input.reshape(1, SEQUENCE_LENGTH, 1)

scaled_prediction = model.predict(model_input, verbose=0).flatten()[0]
prediction = unscale_values(scaled_prediction, mean, standard_deviation)

print("Input temperatures:", new_temperatures.astype(int).tolist())
print(f"Predicted next temperature: {prediction:.2f}")

## What This Notebook Teaches

Key lessons:

1. Time-series data should be split in order, not shuffled.
2. Scaling should be fitted on training data only.
3. Test evaluation should use future data only.
4. A model can have low training loss but still struggle to extrapolate.
5. For this tiny upward-trend example, `SimpleRNN(..., activation="relu")` works better than the default `tanh` activation.

The scripts in `src/` use the same workflow shown here.